# 02 · The shock table

**Question for this notebook:** what does the comparison actually look like, shot by shot, before any effect is estimated?

1. What is in the table
2. Does xG weighting make goals and controls comparable?
3. Were scorers already different before the shot? (an event study)
4. One shot on the pitch

Needs `make shocks` to have run for this config.

In [ ]:
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from confidence_shocks.config import ROOT, load_config

# Which run to look at. Switch to leagues_2015_16.yaml for the main analysis.
CONFIG = ROOT / "configs" / "default.yaml"
cfg = load_config(CONFIG)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S")
pd.set_option("display.max_columns", 40)
print(cfg.name, "|", ", ".join(f"{c['name']} {c['season']}" for c in cfg.competitions))

In [ ]:
from confidence_shocks.estimate import select_controls
from confidence_shocks.features import KIND, METRICS

shocks = pd.read_parquet(cfg.shocks_path)
table = select_controls(shocks, cfg.controls)
print(f"{len(shocks)} shots kept, {len(table)} after keeping goals + '{cfg.controls}' controls")
shocks["outcome"].value_counts()

## 1 · What is in the table

One row per open-play shot where the shooter was on the pitch for the whole window either side. For every metric there are three windows for the shooter (`_prepre`, `_pre`, `_post`) and the same three for his teammates (`_mates_...`, per teammate). `_prepre` is one window further back and feeds the placebo test; it is empty when the shooter came on too late.

In [ ]:
show = ["player", "clock", "score_diff", "xg", "outcome",
        "receipts_prepre", "receipts_pre", "receipts_post", "receipts_mates_pre", "receipts_mates_post"]
table[show].head(10)

In [ ]:
print("share of rows with the earlier (placebo) window:", round(table["receipts_prepre"].notna().mean(), 2))
print("shots per player:", table.groupby("player").size().describe().round(1).to_dict())

The estimator also uses each shooter's **usual level** (his average before-shot level in his *other* matches). With only a few matches per player, as in a tournament, that norm is noisy; full league seasons are where it becomes informative.

## 2 · Does xG weighting make goals and controls comparable?

Goals have higher xG than saved shots. Weighting each control by `xg / (1 - xg)` should make the two xG distributions match.

In [ ]:
goal = table["goal"].to_numpy()
xg = table["xg"].clip(0.01, 0.99).to_numpy()
weight = np.where(goal, 1.0, xg / (1 - xg))
bins = np.linspace(0, 0.8, 17)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, weights, label in [(axes[0], None, "unweighted"), (axes[1], weight, "controls weighted")]:
    w_goal = None if weights is None else weights[goal]
    w_ctrl = None if weights is None else weights[~goal]
    ax.hist(xg[goal], bins, weights=w_goal, density=True, alpha=0.6, label="goals")
    ax.hist(xg[~goal], bins, weights=w_ctrl, density=True, alpha=0.6, label="controls")
    ax.set_title(label)
    ax.set_xlabel("xG")
axes[0].legend()
plt.show()

## 3 · Were scorers already different before the shot?

The 2022 World Cup run showed scorers were *less* involved than controls in the 15 minutes before shooting, and every after-minus-before "effect" pointed the opposite way to that gap: regression to the mean. An event study shows it directly: the shooter's average activity in 5-minute bins around the shot, goals against weighted controls.

If the lines separate only *after* zero, the comparison is clean. If they already differ before zero, the before-shot level has to be adjusted for (which `method: ancova` does), and the placebo check in `checks.csv` will say whether that is enough.

In [ ]:
METRIC = "receipts"           # try "passes", "carries", "dribbles"
EVENT_TYPE = {"receipts": "Ball Receipt*", "passes": "Pass", "carries": "Carry",
              "pressures": "Pressure", "dribbles": "Dribble", "shots": "Shot"}[METRIC]
EDGES = np.arange(-30, 16, 5)  # bins from 30 minutes before to 15 after

# Only shots where the shooter was on for the full 30 minutes before.
study = table[table["receipts_prepre"].notna()]
counts = np.zeros((len(study), len(EDGES) - 1))
for match_id, rows in study.groupby("match_id"):
    events = pd.read_parquet(cfg.events_dir / f"{match_id}.parquet", columns=["player", "type", "clock"])
    events = events[events["type"] == EVENT_TYPE]
    for i, shot in zip(study.index.get_indexer(rows.index), rows.itertuples()):
        times = events.loc[events["player"] == shot.player, "clock"].to_numpy() - shot.clock
        counts[i] = np.histogram(times, EDGES)[0]

g = study["goal"].to_numpy()
xg_s = study["xg"].clip(0.01, 0.99).to_numpy()
w = np.where(g, 1.0, xg_s / (1 - xg_s))
centres = EDGES[:-1] + 2.5

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(centres, np.average(counts[g], axis=0, weights=w[g]), "o-", label=f"goals (n={g.sum()})")
ax.plot(centres, np.average(counts[~g], axis=0, weights=w[~g]), "o-", label=f"controls (n={(~g).sum()})")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("minutes from the shot")
ax.set_ylabel(f"{METRIC} per 5 minutes")
ax.set_title(f"{cfg.name}: shooter's {METRIC} around the shot")
ax.legend()
plt.show()

Note the shot itself sits in the bin just before zero, and the minutes right after a goal include the celebration and kick-off, so the bins touching zero are not comparable between groups. The windows used by the estimator start and stop at the shot for the same reason; a robustness check in Phase 4 drops the first few minutes after it.

## 4 · One shot on the pitch

The shooter's passes before (left) and after (right) one goal. Pick another row with `CASE`.

In [ ]:
from mplsoccer import Pitch

goals = table[table["goal"]].sort_values("xg")
CASE = len(goals) // 2           # a middling-xG goal; change to look at others
shot = goals.iloc[CASE]

events = pd.read_parquet(cfg.events_dir / f"{shot['match_id']}.parquet")
mine = events[(events["player"] == shot["player"]) & (events["type"] == "Pass")]
w_min = cfg.window_minutes
windows = {
    f"{w_min:g} min before": mine[(mine["clock"] >= shot["clock"] - w_min) & (mine["clock"] < shot["clock"])],
    f"{w_min:g} min after": mine[(mine["clock"] > shot["clock"]) & (mine["clock"] <= shot["clock"] + w_min)],
}

pitch = Pitch(pitch_type="statsbomb", line_zorder=2)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (label, passes) in zip(axes, windows.items()):
    pitch.draw(ax=ax)
    done = passes["pass_outcome"].isna()
    pitch.arrows(passes.loc[done, "x"], passes.loc[done, "y"], passes.loc[done, "end_x"],
                 passes.loc[done, "end_y"], ax=ax, width=1.5, color="tab:blue", label="completed")
    pitch.arrows(passes.loc[~done, "x"], passes.loc[~done, "y"], passes.loc[~done, "end_x"],
                 passes.loc[~done, "end_y"], ax=ax, width=1.5, color="tab:red", label="failed")
    ax.set_title(f"{label}: {len(passes)} passes")
axes[0].legend(loc="lower left")
fig.suptitle(f"{shot['player']}, goal at minute {shot['clock']:.0f} (xG {shot['xg']:.2f})")
plt.show()